2-3 Tree Implementation:

A 2-3 tree is a balanced search tree in which
- 2 nodes have one key and two children
- All the leaf nodes lie at the same depth

The tree is balanced by splitting 'overfull' nodes (temporary nodes with 3 keys and 4 children) into two 2 nodes, and passing the middle key up the tree. The tree depth can only be increased by a split operation at the root node: this is how the tree maintains balance.
Splitting is a constant time operation and only acts locally in the structure.

Searching a 2-3 tree of N nodes is an O(logN) operation; inserting a node into a 2-3 tree of size N is also an O(logN) operation. The depth of the tree ranges from log N / log 2 (log base 2 of N), to log N / log 3 (log base 3 of N).

In [ ]:

'''
2-3 Tree Implementation:

A 2-3 tree is a balanced search tree in which
- 2 nodes have one key and two children
- 3 nodes have two keys and three children
- All the leaf nodes lie at the same depth

The tree is balanced by splitting 'overfull' nodes (temporary nodes with 3 keys and 4 children) into two 2 nodes, and passing the middle key up the tree. The tree depth can only be increased by a split operation at the root node: this is how the tree maintains balance.
Splitting is a constant time operation and only acts locally in the structure.

Searching a 2-3 tree of N nodes is an O(logN) operation; inserting a node into a 2-3 tree of size N is also an O(logN) operation. The depth of the tree ranges from log N / log 2 (log base 2 of N), to log N / log 3 (log base 3 of N).
'''

class TwoThreeNode:
  '''A node in a 2-3 tree.

  Attributes:
    keys: A sorted list of one or two keys (can have three keys but only temporarily)
    children: A list of zero (leaf), two (2-node) or three (3-node) child nodes
  '''
  def __init__(self, keys=None, children=None):
    self.keys = keys or []
    self.children = children or []
  def is_leaf(self):
    return len(self.children) == 0

class TwoThreeTree:
  '''A 2-3 balanced search tree with 'search' and 'insert' operations.
  '''
  def __init__(self, root=None):
    self.root = root

  def searchElement(self, element):
    '''Returns true if and only if the 2-3 tree contains a specified element. Delegates work to recursive helper, 'search'.

    Args:
        node (Node): the root of the tree/subtree to search
        element (_): the element to search for
    '''
    if self.root is None:
      return False
    return self.__search(self.root, element)
  def __search(self, node, element):
    '''Returns true if and only if the subtree rooted at a given node contains a specified element.

    Args:
        node (Node): the root of the tree/subtree to search
        element (_): the element to search for
    
    '''

    if node is None:
      return False
    
    # Checks if the element is in this node
    if element in node.keys:
      return True
    
    # Leaf reached without finding the element -> the element is not in the tree
    if node.is_leaf():
      return False
    
    # Pick the correct child to descend the tree to
    # The index of the first key greater than the element is the index of the correct child
    # (If no key is greater, then descend to the rightmost child)
    child_index = len(node.keys)
    for i, key in enumerate(node.keys):
      if element < key:
        child_index = i
        break

    return self.__search(node.children[child_index], element)
  
  def insertElement(self, element):
    '''Insert a element into a 2-3 tree, keeping the balance in check. Delegates work to the recursive _insert method.

    Args:
        element (_): the element to insert
    Returns:
        True if and only if the element is successfully inserted into the tree; False otherwise (if there are duplicates)
    '''
    # If the tree is empty, then create a single leaf root
    if self.root is None:
      self.root = TwoThreeNode([element])
      return True
    
    result = self.__insertElement(self.root, element)
    
    # Result is false -> element already in the tree -> return false
    if result is False:
      return False
    
    # If the root split, then create a new one, one level higher
    # This is the only way the tree can grow taller
    if result is not None:
      middle_key, left_child, right_child = result
      self.root = TwoThreeNode([middle_key], [left_child, right_child])
    
    return True 
  
  def __insertElement(self, node, element):
    '''Insert a element into a 2-3 subtree rooted at a given node, keeping the balance in check. This is a recurisve helper method for insert.

    Args:
        node (Node): the root node of the subtree
        element (_): the element to insert

    Returns:
        None : if the insertion is absorbed by the node without causing a split operation
        middle_key, left_child, right_child (_, Node, Node) : if the node split, the calling method must absorb the key being passed up, and the left and right children
        False : if the element being inserted is already in the tree
    '''
    # Element already in tree -> return false
    if element in node.keys:
      return False
    
    # Base case - leaf node
    if node.is_leaf():

      node.keys.append(element)
      node.keys.sort()

      if len(node.keys) <= 2: # Still a valid 2- or 3- node
        return None
      
      # Overfull (has 3 keys) -> split and pass the middle key up the call chain
      middle_key = node.keys[1]
      left_child = TwoThreeNode([node.keys[0]])
      right_child = TwoThreeNode([node.keys[2]])
      return (middle_key, left_child, right_child)
          
    # Recursive case - 2- or 3- (non-leaf) node
    # Find the right child to recurse to (same logic as in search)
    child_index = len(node.keys)
    for i, key in enumerate(node.keys):
      if element < key:
        child_index = i
        break

    result = self.__insertElement(node.children[child_index], element)
    
    # result False -> duplicate data -> propagate this up the call stack 
    if result is False:
      return False
    if result is None: # Child absorbed the insert without splitting - so do nothing
      return None   
    
    # A child split, so absorb the recieved key and new children
    middle_key, left_child, right_child = result
    node.keys.append(middle_key)
    node.keys.sort()
    # Replace the child at child_index with the two new children
    node.children = node.children[:child_index] + [left_child, right_child] + node.children[child_index+1:]

    if len(node.keys) <= 2: # Still a valid 2- or 3- node - so do not initiate a split
      return None
    # Overfull (has 3 keys) -> split and pass (as before) but attach children
    # left_child node gets the first two children, right_child node get the last two
    middle_key = node.keys[1] 
    left_child = TwoThreeNode([node.keys[0]], node.children[:2])
    right_child = TwoThreeNode([node.keys[2]], node.children[2:])
    return (middle_key, left_child, right_child)

In [ ]:
'''Extremely basic test
'''

from random import shuffle 
bst = TwoThreeTree()


shuffled_even_integers = [i for i in range(0, 100, 2)]
shuffle(shuffled_even_integers)

for i in shuffled_even_integers:
  bst.insert(i)

positives = [] # Expect all even integers 0 - 100
negatives = [] # Expect all odd integers 0 - 100
integer_range = [i for i in range(100)]
for i in integer_range:
  if bst.search(i):
    positives += [i]
  else:
    negatives += [i]

print("The positives were: ", positives)
print("The negatives were: ", negatives)

# Note: I wrote this to verify my implementation works - we need to agree on a rigorous testing framework

The positives were:  [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98]
The negatives were:  [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55, 57, 59, 61, 63, 65, 67, 69, 71, 73, 75, 77, 79, 81, 83, 85, 87, 89, 91, 93, 95, 97, 99]
